# proto-REINFORCE, notebook 1: Old Bessie

On page 3 of Old Bessie, proto-REINFORCE learned to manage the van: bets picked at random, 1,000 runs, their
average $G_1$, then one small change of one bet at a time. In this notebook you write it in Python, one small
piece at a time, and your code replays the page: the start at -\$172.24, update 1 to \$73.57, and 300 updates.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. It takes about 35
minutes.

Start with the two folded cells below: Old Bessie's kit and the checks.

In [ ]:
#@title Old Bessie's kit (run me first) { display-mode: "form" }
# Old Bessie's kit. Run this cell first; you do not need to read it.
from math import exp            # e to the power: exp(0.4) is 1.4918...
import math as _kit_math

STATES = ['HEALTHY', 'WORN', 'SHAKY', 'FAILING']
ACTIONS = ['RUN', 'SERVICE', 'REPLACE']
HEALTHY, WORN, SHAKY, FAILING = 0, 1, 2, 3       # the rows of a table: the van's states
RUN, SERVICE, REPLACE = 0, 1, 2                  # the columns of a table: the actions
GAMMA = 0.9                # a week later, a dollar counts 0.9 times as much
WEEKS = 13                 # the weeks of a quarter: one run
RUNS = 1000                # the quarters of every average on page 3

# The van of the arcade, as on page 1 (bessie.js): the week's money and the chances of what happens
_KIT_PROFIT = [95, 72, 40, 16]          # RUN: the route's money, by state
_KIT_BREAKDOWN = [0.02, 0.08, 0.28, 0.55]    # RUN: the chance of a breakdown, by state
_KIT_TOW = 280                       # a breakdown costs this on top, and the van is FAILING next week
_KIT_WEAR = [[0.35, 0.55, 0.1], [0.3, 0.55, 0.15], [0.45, 0.55, 0.0], [0.65, 0.35, 0.0]]     # RUN without a breakdown: wears 0, 1 or 2 levels
_KIT_SHOP = 50                       # SERVICE: a week in the garage
_KIT_REPAIR = [[1.0, 0.0, 0.0], [0.05, 0.7, 0.25], [0.45, 0.5, 0.05], [0.55, 0.42, 0.03]]     # SERVICE: repairs 0, 1 or 2 levels
_KIT_NEW_VAN = 130                   # REPLACE: a new van, HEALTHY next week
_KIT_START_BETS = [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]   # page 3's bets at random
_KIT_BEST_LINE = 244.03          # the best strategy's average G1 on the quarters of update 300 (page 3)


def _kit_outcomes(state, action):
    """Every outcome of a week, in the order of the page: [running chance, next state, money]."""
    out = []
    if action == RUN:
        out.append([_KIT_BREAKDOWN[state], FAILING, _KIT_PROFIT[state] - _KIT_TOW])
        for d in range(3):
            p = (1 - _KIT_BREAKDOWN[state]) * _KIT_WEAR[state][d]
            if p > 0:
                out.append([p, min(FAILING, state + d), _KIT_PROFIT[state]])
    elif action == SERVICE:
        for d in range(3):
            p = _KIT_REPAIR[state][d]
            if p > 0:
                out.append([p, max(HEALTHY, state - d), -_KIT_SHOP])
    else:
        out.append([1, HEALTHY, -_KIT_NEW_VAN])
    running = 0
    for o in out:
        running = running + o[0]
        o[0] = running
    return out


_KIT_WEEK = []
for _s in range(4):
    _KIT_WEEK.append([])
    for _a in range(3):
        _KIT_WEEK[_s].append(_kit_outcomes(_s, _a))


def week(state, action, u):
    """One week of Old Bessie. In this state, with this action, the number u (between 0 and 1) picks what happens,
    with the chances of page 1. Gives the van's state next week and the week's money:
        next_state, money = week(HEALTHY, RUN, 0.5)"""
    try:
        outcomes = _KIT_WEEK[state][action]
    except (IndexError, TypeError):
        raise ValueError("week needs a state from 0 to 3 and an action from 0 to 2; it got the state %r and the "
                         "action %r" % (state, action))
    for running, next_state, money in outcomes:
        if u < running:
            return next_state, money
    return outcomes[-1][1], outcomes[-1][2]


def roll_dice(seed, runs):
    """The dice of many quarters, as on the page: dice[run][week] holds two numbers between 0 and 1, the first to
    pick the action, the second for the van's luck. The same seed always rolls the same numbers."""
    state = seed & 0xFFFFFFFF
    dice = []
    for r in range(runs):
        quarter = []
        for w in range(WEEKS):
            pair = []
            for k in range(2):
                state = (state + 0x6D2B79F5) & 0xFFFFFFFF
                t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
                t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
                pair.append(((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296)
            quarter.append(pair)
        dice.append(quarter)
    return dice


_KIT_DICE = {}


def dice_of_update(number):
    """The 1,000 quarters of dice of an update of page 3: the page's first dice (seed 12) for the start,
    update 0, and for updates 1 and 2, fresh dice (seed 2000 + number) for every later update."""
    seed = 12 if number <= 2 else 2000 + number
    if seed not in _KIT_DICE:
        if len(_KIT_DICE) >= 2:
            _KIT_DICE.clear()
        _KIT_DICE[seed] = roll_dice(seed, RUNS)
    return _KIT_DICE[seed]


def _kit_numbers(seed):
    state = [seed & 0xFFFFFFFF]

    def next_number():
        state[0] = (state[0] + 0x6D2B79F5) & 0xFFFFFFFF
        s = state[0]
        t = ((s ^ (s >> 15)) * (1 | s)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    return next_number


_KIT_PICKS = []


def picked_bet(number):
    """The bet that update number (1, 2, 3, ...) picks at random, as on the page: [state, action]."""
    if isinstance(number, bool) or not isinstance(number, int) or number < 1:
        raise ValueError("picked_bet needs the number of an update: 1, 2, 3 and so on; it got %r" % (number,))
    if len(_KIT_PICKS) < number:
        next_number = _kit_numbers(134623)
        del _KIT_PICKS[:]
        while len(_KIT_PICKS) < number:
            k = int(next_number() * 12)              # one of the 12 bets, each as likely as the others
            _KIT_PICKS.append([k // 3, k % 3])      # its row, the state, and its column, the action
    return list(_KIT_PICKS[number - 1])


def start_bets():
    """Page 3's table of bets picked at random, between -10 and 10: a fresh copy each time."""
    return copy_bets(_KIT_START_BETS)


def copy_bets(bets):
    """A copy of a table of bets: change the copy, and the table stays as it was."""
    copy = []
    for row in bets:
        copy.append(list(row))
    return copy


def _kit_is_table(table, columns=3):
    if not isinstance(table, list) or len(table) != len(STATES):
        return False
    for row in table:
        if not isinstance(row, list) or len(row) != columns:
            return False
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, float)) or x != x:
                return False
    return True


def _kit_short(value):
    text = repr(value)
    return text if len(text) <= 60 else text[:57] + "..."


def _kit_print_table(table, places, mark):
    head = " " * 9
    for name in ACTIONS:
        head += "%10s" % name
    if mark:
        head += "    likeliest"
    print(head)
    for s in range(len(STATES)):
        line = "%-9s" % STATES[s]
        best = 0
        for a in range(len(ACTIONS)):
            line += "%10.*f" % (places, table[s][a])
            if table[s][a] > table[s][best]:
                best = a
        if mark:
            line += "    " + ACTIONS[best]
        print(line)


def show_bets(bets):
    """Print a table of bets: a row per state, a column per action, two decimals."""
    if not _kit_is_table(bets):
        print("⏳ No table of bets to show yet: show_bets needs four rows of three numbers. It got %s." % _kit_short(bets))
        return
    _kit_print_table(bets, 2, False)


def show_chances(table):
    """Print a table of chances (the policy): a row per state, a column per action, two decimals, and the likeliest
    action of each row."""
    if not _kit_is_table(table):
        print("⏳ No table of chances to show yet: show_chances needs four rows of three numbers. It got %s."
              % _kit_short(table))
        return
    _kit_print_table(table, 2, True)


def _kit_money(x):
    """-$172.24, $73.57"""
    text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def _kit_axis(x):
    """-$150, $250"""
    text = "$%d" % abs(int(round(x)))
    return "-" + text if x < 0 else text


def plot_rewards(points, best=_KIT_BEST_LINE, label="best strategy"):
    """The reward plot of page 3, as text: across, the number of updates; up, the average G1 after them. o is a
    point (one column for every few updates when there are many), and the row of dashes marks the best strategy's
    average G1 on the quarters of the last update of the page."""
    ok = isinstance(points, list) and len(points) >= 2
    if ok:
        for x in points:
            if isinstance(x, bool) or not isinstance(x, (int, float)) or x != x:
                ok = False
    if not ok:
        print("⏳ No reward plot yet: plot_rewards needs a list of at least two averages. It got %s." % _kit_short(points))
        return
    n = len(points) - 1                      # the updates
    columns = min(n + 1, 61)
    where = []                               # the update that each column shows
    for c in range(columns):
        where.append(int(round(c * n / (columns - 1))))
    values = []
    for i in where:
        values.append(points[i])
    low, high = min(values + [best]), max(values + [best])
    step = 10
    for size in (10, 25, 50, 100, 250, 500, 1000):
        step = size
        if (high - low) / size <= 18:
            break
    top = _kit_math.ceil(high / step) * step
    bottom = _kit_math.floor(low / step) * step
    rows = int(round((top - bottom) / step)) + 1
    grid = []
    for r in range(rows):
        grid.append([" "] * columns)
    best_row = int(round((top - best) / step))
    for c in range(columns):
        grid[best_row][c] = "-"
    for c in range(columns):
        r = int(round((top - values[c]) / step))
        grid[r][c] = "o"
    every = max(1, int(round(n / (columns - 1)))) if columns > 1 else 1
    print("The reward plot: the average G1 after each update" + (" (one o for every %d updates)" % every if every > 1
                                                                 else ""))
    print()
    for r in range(rows):
        line = "%8s |" % _kit_axis(top - r * step) + "".join(grid[r])
        if r == best_row:
            line += "  %s, %s" % (label, _kit_money(best))
        print(line.rstrip())
    print(" " * 9 + "+" + "-" * columns)
    ticks = 50 if n >= 200 else (10 if n >= 40 else 1)
    marks = [" "] * (columns + 6)
    for t in range(0, n + 1, ticks):
        c = int(round(t * (columns - 1) / n)) if n else 0
        text = str(t)
        for j in range(len(text)):
            if c + j < len(marks):
                marks[c + j] = text[j]
    print(" " * 10 + "".join(marks).rstrip() + "   updates")
    print("first point %s, last point %s" % (_kit_money(points[0]), _kit_money(points[-1])))

In [ ]:
#@title The checks of notebook 1 (run me too) { display-mode: "form" }
# The checks of notebook 1 and play. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

_N1_TESTS = {'cumulative_reward': [{'about': "page 2's run (RUN until it fails, 13 weeks)", 'args': [[95, 72, 40, 40, 40, -130, 95, -240, -130, 95, 95, 72, 40]], 'kind': 'number', 'want': 154.39614434222, 'tol': 1e-06, 'keep': [0]}, {'about': 'a run of two weeks, $100 each', 'args': [[100, 100]], 'kind': 'number', 'want': 190.0, 'tol': 1e-06}, {'about': 'a run of three weeks, $10, -$20 and $30', 'args': [[10, -20, 30]], 'kind': 'number', 'want': 16.3, 'tol': 1e-06}], 'chances': [{'about': "HEALTHY's bets on page 3, 0.40, -0.60, 0.10", 'args': [[0.4, -0.6, 0.1]], 'kind': 'numbers', 'want': [0.47422635216524245, 0.174458125423321, 0.3513155224114366], 'tol': 1e-09, 'keep': [0]}, {'about': "SHAKY's bets on page 3, -0.40, 0.10, 10.00", 'args': [[-0.4, 0.1, 10]], 'kind': 'numbers', 'want': [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], 'tol': 1e-09, 'keep': [0]}, {'about': 'three equal bets, 0, 0 and 0', 'args': [[0, 0, 0]], 'kind': 'numbers', 'want': [0.3333333333333333, 0.3333333333333333, 0.3333333333333333], 'tol': 1e-09}, {'about': 'a row of two bets, 1 and 2', 'args': [[1, 2]], 'kind': 'numbers', 'want': [0.2689414213699951, 0.7310585786300049], 'tol': 1e-09}], 'pick_action': [{'about': "HEALTHY's chances on page 3 and u = 0.2882", 'args': [[0.47422635216524245, 0.174458125423321, 0.3513155224114366], 0.2881509135477245], 'kind': 'action', 'want': 0, 'tol': 0}, {'about': "HEALTHY's chances on page 3 and u = 0.5", 'args': [[0.47422635216524245, 0.174458125423321, 0.3513155224114366], 0.5], 'kind': 'action', 'want': 1, 'tol': 0}, {'about': "HEALTHY's chances on page 3 and u = 0.9", 'args': [[0.47422635216524245, 0.174458125423321, 0.3513155224114366], 0.9], 'kind': 'action', 'want': 2, 'tol': 0}, {'about': 'the chances 0.5 and 0.5 and u = 0.5', 'args': [[0.5, 0.5], 0.5], 'kind': 'action', 'want': 1, 'tol': 0}, {'about': 'the chances 0, 1 and 0 and u = 0', 'args': [[0.0, 1.0, 0.0], 0.0], 'kind': 'action', 'want': 1, 'tol': 0}, {'about': 'the chances 0.25, 0.25 and 0.5 and u = 0.3', 'args': [[0.25, 0.25, 0.5], 0.3], 'kind': 'action', 'want': 1, 'tol': 0}], 'play_run': [{'about': "run 1 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 0]}], 'kind': 'number', 'want': -349.6474260845501, 'tol': 1e-06, 'keep': [0]}, {'about': "run 2 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 1]}], 'kind': 'number', 'want': 322.030256997182, 'tol': 1e-06, 'keep': [0]}, {'about': "run 3 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 2]}], 'kind': 'number', 'want': -313.2251107889801, 'tol': 1e-06, 'keep': [0]}, {'about': "run 4 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 3]}], 'kind': 'number', 'want': -90.93935904250003, 'tol': 1e-06, 'keep': [0]}, {'about': "run 5 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 4]}], 'kind': 'number', 'want': -73.59923716898001, 'tol': 1e-06, 'keep': [0]}, {'about': "run 6 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 5]}], 'kind': 'number', 'want': -500.2791715542102, 'tol': 1e-06, 'keep': [0]}, {'about': "run 7 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 6]}], 'kind': 'number', 'want': 146.37524608899503, 'tol': 1e-06, 'keep': [0]}, {'about': "run 8 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 7]}], 'kind': 'number', 'want': -222.383623910755, 'tol': 1e-06, 'keep': [0]}, {'about': "run 9 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 8]}], 'kind': 'number', 'want': -93.46884060954997, 'tol': 1e-06, 'keep': [0]}, {'about': "run 10 of page 3's start (the bets at random)", 'args': [[[0.47422635216524245, 0.174458125423321, 0.3513155224114366], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 9]}], 'kind': 'number', 'want': -230.3546292892301, 'tol': 1e-06, 'keep': [0]}, {'about': "run 1 after update 1 (HEALTHY's bet of RUN at 2.14)", 'args': [[[0.8371009448688169, 0.0540518983947786, 0.10884715673640454], [0.524420465275956, 0.47451525976551256, 0.0010642749585314882], [3.043003012994202e-05, 5.01706379432812e-05, 0.9999193993319269], [0.37442922164689685, 0.4573288840552853, 0.16824189429781772]], {'run': [12, 0]}], 'kind': 'number', 'want': 91.21237641544997, 'tol': 1e-06}, {'about': "page 2's run, with RUN until it fails as a table of chances", 'args': [[[1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]], {'run': [41, 0]}], 'kind': 'number', 'want': 154.39614434222, 'tol': 1e-06}], 'average_reward': [{'about': "page 3's start: the bets at random and the 1,000 runs of its dice", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], {'dice': [12, 0, 1000]}], 'kind': 'number', 'want': -172.24, 'tol': 1e-06, 'keep': [0]}, {'about': "the first 10 runs of page 3's start", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], {'dice': [12, 0, 10]}], 'kind': 'number', 'want': -140.55, 'tol': 1e-06, 'keep': [0]}, {'about': "the table after update 1 and the start's 1,000 runs", 'args': [[[2.14, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], {'dice': [12, 0, 1000]}], 'kind': 'number', 'want': 73.57, 'tol': 1e-06, 'keep': [0]}], 'slope': [{'about': 'update 1 of page 3 (the bets at random, HEALTHY and RUN, a hop of 0.01)', 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, {'dice': [12, 0, 1000]}, 0.01], 'kind': 'number', 'want': 174, 'tol': 1e-09, 'keep': [0]}, {'about': 'update 2 of page 3 (HEALTHY and REPLACE, after update 1)', 'args': [[[2.14, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 2, {'dice': [12, 0, 1000]}, 0.01], 'kind': 'number', 'want': -13, 'tol': 1e-09, 'keep': [0]}, {'about': 'the bets at random, HEALTHY and RUN, the first 10 runs and a hop of 0.5', 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, {'dice': [12, 0, 10]}, 0.5], 'kind': 'number', 'want': 258, 'tol': 1e-09, 'keep': [0]}], 'update_bet': [{'about': 'update 1 of page 3 (0.40, nabla 174, alpha 0.01)', 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, 174, 0.01], 'kind': 'number', 'want': 2.14, 'tol': 1e-09, 'table': {'arg': 0, 'want': [[2.14, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}, {'about': 'update 2 of page 3 (HEALTHY and REPLACE, 0.10, nabla -13)', 'args': [[[2.14, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 2, -13, 0.01], 'kind': 'number', 'want': -0.03, 'tol': 1e-09, 'table': {'arg': 0, 'want': [[2.14, -0.6, -0.03], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}, {'about': 'WORN and SERVICE (-0.80, nabla 10, alpha 0.5)', 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 1, 1, 10, 0.5], 'kind': 'number', 'want': 4.2, 'tol': 1e-09, 'table': {'arg': 0, 'want': [[0.4, -0.6, 0.1], [-0.7, 4.2, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}], 'proto_reinforce': [{'about': 'the first 3 updates of page 3', 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 3, 0.01, 0.01], 'kind': 'numbers', 'want': [-172.24, 73.57, 80.98, 96.7], 'tol': 1e-09, 'table': {'arg': 0, 'want': [[2.14, -0.6, -0.03], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10.0], [0.4, 0.6, -0.4]]}}, {'about': '1 update with alpha = 0.02', 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 1, 0.01, 0.02], 'kind': 'numbers', 'want': [-172.24, 146.29], 'tol': 1e-09, 'table': {'arg': 0, 'want': [[3.88, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}]}
_N1_SLIPS = {'cumulative_reward': [{'answers': [184, 200, 20], 'hint': 'each week counts 0.9 times as much as the week before: multiply the money by discount before you add it'}, {'answers': [138.95652990799798, 171.0, 14.670000000000002], 'hint': 'week 1 counts fully: add discount * money first, and shrink discount after'}, {'answers': [175.10000000000002, 190.0, 19.0], 'hint': 'the discount shrinks every week: discount = discount * GAMMA, so that week 3 counts 0.9 x 0.9'}, {'answers': [11.297181459240006, 90.0, 24.3], 'hint': 'total keeps the weeks before: total = total + discount * money'}], 'chances': [{'answers': [[-4.000000000000003, 6.000000000000003, -1.0000000000000007], [-0.04123711340206186, 0.010309278350515465, 1.0309278350515465], None, [0.3333333333333333, 0.6666666666666666]], 'hint': 'first e to the power of each bet, exp(bet): that makes every chance positive'}, {'answers': [[1.4918246976412703, 0.5488116360940265, 1.1051709180756477], [0.6703200460356393, 1.1051709180756477, 22026.465794806718], [1.0, 1.0, 1.0], [2.718281828459045, 7.38905609893065]], 'hint': 'divide each power by the total of the powers, so that the chances add up to 1'}, {'answers': [[-14.91824697641271, -5.488116360940268, -11.051709180756482], [0.06910515938511747, 0.11393514619336575, 2270.769669567703], None, [0.9060939428196817, 2.46301869964355]], 'hint': 'the total adds up the powers, the numbers in the list powers'}], 'pick_action': [{'answers': [0, 1, 2, 0, 0, 1], 'hint': 'compare with u < running: the number at the very end of a slice belongs to the next slice'}, {'answers': [0, 2, 2, 1, 1, 2], 'hint': 'compare u with the running total, which adds the chances up action by action: running = running + row_of_chances[action]'}, {'answers': [2, 0, 0, 1, 2, 0], 'hint': 'the roulette stops at the first action whose running total passes u: if u < running'}], 'play_run': [{'answers': [-473.4444018028181, -181.5405538294302, -327.4068552922301, 197.57116402115005, 452.010154803395, -266.36182540781806, -192.01101515698008, 8.25591962551993, -14.263881332530019, -405.18838818185003, -242.282978591818, 154.39614434222], 'hint': "the first number of the week, numbers[0], picks the action; the second, numbers[1], is the van's luck, the u of week"}, {'answers': [-165.14091166625496, 328.024219836245, 258.730967019245, -210.1989339085001, 12.439349786245035, -266.27676573100007, 166.54173837399506, -305.86526303075505, 69.49866101900008, -207.79130901973008, 394.55313628374506, 708.5227463087451], 'hint': 'next week starts where this week ends: state = next_state'}, {'answers': [-296.20170509746004, 130.33364244618198, -22.93265748721, -230.89893390850008, -104.66648618189004, -281.51401356337016, 127.03032129699504, -305.86526303075505, 19.470492308930048, -235.87929447223007, -170.47969583746004, -370.90272252114426], 'hint': 'the roulette uses the row of the state the van is in this week: policy[state]'}], 'average_reward': [{'answers': [-172.24, -1.41, 73.57], 'hint': 'divide by the number of runs in dice, len(dice): the check also tries 10 runs'}, {'answers': [-172243.3, -1405.49, 73569.34], 'hint': 'the average is the total divided by the number of runs, len(dice)'}, {'answers': [-439.56, -414.04, 54.24], 'hint': 'play_run takes the table of chances, policy, which the first line makes from the bets'}, {'answers': [-13249.48, -108.11, 5659.18], 'hint': 'divide by the number of runs, len(dice): the 13 weeks of a run are already inside its G1'}], 'slope': [{'answers': [0, 0, 64], 'hint': 'nabla came out 0: After must be the average G1 of the hopped table, and nabla is the change divided by the hop, change / hop'}, {'answers': [2, 0, 129], 'hint': 'divide the change by the hop, the side across of the triangle: nabla = change / hop'}, {'answers': [174, -13, 12890], 'hint': 'divide by hop, the input of slope: the check also tries a hop of 0.5'}, {'answers': [-9244, 449, 107], 'hint': 'the hop adds to the bet: hopped[state][action] + hop'}], 'update_bet': [{'answers': [-1.34, 0.23, -5.8], 'hint': 'here the average G1 has to go up: w + alpha * nabla (in weekend 2 a loss had to go down, w - alpha * nabla)'}, {'answers': [174.4, -12.9, 9.2], 'hint': 'alpha keeps each move small: w + alpha * nabla'}, {'answers': [1.74, -0.13, 5.0], 'hint': 'the new bet starts from the bet as it is: w + alpha * nabla'}, {'answers': [2.14, -0.03, -0.7], 'hint': 'use alpha, the input of update_bet: the check also tries alpha = 0.5'}], 'proto_reinforce': [{'answers': [[-172.24, 73.57, 80.98, 96.7], [-172.24, 104.91]], 'hint': 'slope takes the hop and update_bet takes alpha: the check also tries alpha = 0.02'}, {'answers': [[-172.24, 73.57, 80.98, 96.7], [-172.24, 73.57]], 'hint': 'update_bet takes alpha, the input of proto_reinforce: the check also tries alpha = 0.02'}]}
_N1_SUCCESS = {'cumulative_reward': "cumulative_reward works: page 2's run gives G1 = $154.40, as on page 2.", 'chances': "chances works: HEALTHY's bets 0.40, -0.60, 0.10 give the chances 0.47, 0.17, 0.35, and SHAKY's bet of 10.00 makes REPLACE almost certain.", 'pick_action': 'pick_action works: every number u falls in the slice of the right action.', 'play_run': "play_run works: the first ten runs of page 3's start give G1 = -$350, $322, -$313, -$91, -$74, -$500, $146, -$222, -$93, -$230 in whole dollars, as on page 3.", 'average_reward': "average_reward works: the bets at random give the average G1 -$172.24 over the 1,000 runs, page 3's start.", 'slope': "slope works: update 1's bet, HEALTHY and RUN, gets nabla = 174, as on page 3.", 'update_bet': 'update_bet works: 0.40 + 0.01 x 174 = 2.14, as on page 3.', 'proto_reinforce': 'proto_reinforce works: its first three updates give the points -$172.24, $73.57, $80.98, $96.70, as on page 3.'}
_N1_DEFAULT = {'cumulative_reward': 'add discount * money to total, then multiply discount by GAMMA for the next week', 'chances': 'fill powers with exp(bet), add them up into total, then divide each power by total', 'pick_action': "add each action's chance to running, and give back the first action with u < running", 'play_run': 'the action is pick_action(policy[state], numbers[0]), and the state of next week is next_state', 'average_reward': 'add play_run(policy, run_dice) to total for every run, then divide total by len(dice)', 'slope': 'add hop to the bet in hopped, take After as the average G1 of hopped on the same dice, then nabla = change / hop', 'update_bet': 'the new bet is w + alpha * nabla', 'proto_reinforce': 'nabla = slope(bets, state, action, dice, hop), then update_bet(bets, state, action, nabla, alpha)'}
_N1_KEEP_HINT = {'cumulative_reward': 'cumulative_reward reads the list of rewards and leaves it as it is', 'chances': 'chances reads the row of bets and leaves it as it is: the chances go into a new list', 'pick_action': 'pick_action reads the row of chances and leaves it as it is', 'play_run': 'play_run reads the table of chances and leaves it as it is', 'average_reward': 'average_reward reads the table of bets and leaves it as it is', 'slope': 'slope leaves the table of bets as it is: the hop goes on the copy, hopped', 'update_bet': 'update_bet puts the new bet into the table with the given line, bets[state][action] = ...', 'proto_reinforce': 'proto_reinforce changes the table of bets it gets, one bet per update, with update_bet(bets, state, action, nabla, alpha)'}
_N1_TASKS = {'cumulative_reward': 'task A', 'chances': 'task B', 'pick_action': 'task C', 'play_run': 'task D', 'average_reward': 'task E', 'slope': 'task F', 'update_bet': 'task G', 'proto_reinforce': 'task H'}
_N1_NEEDS = {'chances': [], 'cumulative_reward': [], 'pick_action': [], 'table_of_chances': ['chances'], 'play_run': ['pick_action', 'cumulative_reward'], 'average_reward': ['table_of_chances', 'play_run'], 'slope': ['average_reward'], 'update_bet': [], 'proto_reinforce': ['slope', 'update_bet', 'average_reward']}
_N1_LONG = ['proto_reinforce']
_N1_NONE_HINT = {}
_N1_DICE_OF = 'roll_dice'
_N1_GAPS = {}
_N1_WORKING = {}
_N1_DICE = {}


def _n1_num(x):
    """A number with at most 4 decimals: 154.3961, 2.14, 174."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x != x or x in (float("inf"), float("-inf")):
        return repr(x)
    if isinstance(x, int):
        return str(x)
    text = ("%.4f" % x).rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    if text == "0" and x != 0:
        text = "%.2g" % x
    return text


def _n1_text(value):
    """Numbers with at most 4 decimals, lists of them in brackets."""
    if isinstance(value, (list, tuple)):
        parts = []
        for x in value:
            parts.append(_n1_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n1_num(value)


def _n1_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = _n1_text(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 72:
        text = text[:69] + "..."
    return text


def _n1_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _n1_same(kind, got, want, tol):
    """True when got is the right answer want, within tol."""
    try:
        if kind == "number":
            return _n1_is_number(got) and abs(got - want) <= tol
        if kind == "action":
            return _n1_is_number(got) and got == want
        if kind in ("numbers", "table"):
            if not isinstance(got, (list, tuple)) or len(got) != len(want):
                return False
            for k in range(len(want)):
                if not _n1_same("numbers" if kind == "table" else "number", got[k], want[k], tol):
                    return False
            return True
    except Exception:
        return False
    return False


def _n1_copy(value):
    """A deep copy of lists of lists of numbers."""
    if isinstance(value, list):
        out = []
        for x in value:
            out.append(_n1_copy(x))
        return out
    return value


def _n1_dice(seed, first, count):
    """The dice of count runs of a seed from run first, rolled by the kit once per seed and kept."""
    rolled = _N1_DICE.get(seed)
    if rolled is None or len(rolled) < first + count:
        if len(_N1_DICE) > 3:
            _N1_DICE.clear()
        rolled = globals()[_N1_DICE_OF](seed, max(first + count, 10))
        _N1_DICE[seed] = rolled
    return rolled[first:first + count]


def _n1_args(test):
    """Fresh inputs for a test: copies of the lists, and the dice it names."""
    args = []
    for a in test["args"]:
        if isinstance(a, dict) and "dice" in a:
            args.append(_n1_dice(a["dice"][0], a["dice"][1], a["dice"][2]))
        elif isinstance(a, dict) and "run" in a:
            args.append(_n1_dice(a["run"][0], a["run"][1], 1)[0])
        else:
            args.append(_n1_copy(a))
    return args


def _n1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N1_GAPS:
        return _N1_GAPS[code]
    # A gap such as "total = ..." stays in the compiled function; a gap such as "if ...:" only in the source, which
    # Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N1_GAPS[code] = gap
    return gap


def _n1_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n1_key(name, seen=()):
    """The code of the function called name and of the functions it needs: a new version of any of them is checked
    again."""
    if name in seen:
        return ()
    fn = globals().get(name)
    key = [getattr(fn, "__code__", None)]
    for other in _N1_NEEDS.get(name, []):
        key.extend(_n1_key(other, seen + (name,)))
    return tuple(key)


def _n1_call(fn, args):
    """Run fn with its prints silenced: {"answer", "error" (None when the call went through), "line" (the line of
    your code where it happened)}. Never raises."""
    import contextlib, io, traceback
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*args)
    except Exception as error:
        run["error"] = "%s: %s" % (type(error).__name__, error)
        try:
            skip = (_n1_call.__code__.co_filename, globals()[_N1_DICE_OF].__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    return run


def _n1_explain(run):
    """A hint for an error."""
    text = run["error"]
    if "NoneType" in text:
        return 'a value in that line is None: a function it calls gave nothing back (is that function finished, with its return line?), or a name holds None'
    if "ellipsis" in text:
        return "a ... is still in a function that this one calls: finish that function first"
    if "list indices must be integers" in text:
        return 'a place in [ ] must be a whole number: a state or an action, such as policy[state] for the row of a state and row[action] for one of its numbers'
    if "is not subscriptable" in text:
        return "that line uses [ ] on a single number: only a list has places to look up"
    if "is not iterable" in text:
        return ("a single number stands where a list was expected: check the names on the left of each = (update_bet "
                "changes the table itself and gives back only the new bet)")
    if 'week needs' in text:
        return 'the game got a state or an action it does not know: check what pick_action gives back'
    if "ZeroDivisionError" in text:
        return "a division by zero: divide by a number that cannot be 0, such as the total or len(...)"
    if "NameError" in text:
        return "a name in that line is not defined: check its spelling, and run the cells above it first"
    if "can only concatenate" in text or "unsupported operand" in text:
        return "that line mixes a list and a number: take one number of the list with [ ] first"
    if "positional argument" in text:
        return "a function got more or fewer inputs than it takes: compare the call with the function's def line"
    if "out of range" in text:
        return "a place in [ ] is past the end of the list: count from 0, the last place is len(...) - 1"
    return "read that line again, compare it with the hints, run the function's cell, then this check again"


def _n1_failure(name, fn):
    """None when fn passes every test of its task, else (head, what, hint)."""
    tests = _N1_TESTS[name]
    for k in range(len(tests)):
        test = tests[k]
        args = _n1_args(test)
        kept = _n1_copy(args)
        run = _n1_call(fn, args)
        head = "On " + test["about"]
        if run["error"] is not None:
            where = ""
            if run["line"]:
                where = ", in the line  %s " % run["line"]
            return (head, "%s stopped with an error, %s%s" % (name, run["error"], where), _n1_explain(run))
        got = run["answer"]
        if not _n1_same(test["kind"], got, test["want"], test["tol"]):
            if got is None:
                return (head, "%s gave None" % name, _N1_NONE_HINT.get(
                    name, "a function gives its answer with return: keep the return line at its end"))
            hint = _N1_DEFAULT[name]
            for slip in _N1_SLIPS[name]:
                answer = slip["answers"][k]
                if answer is not None and _n1_same(test["kind"], got, answer, test["tol"]):
                    hint = slip["hint"]
                    break
            return (head, "%s should give %s, and yours gives %s" % (name, _n1_short(test["want"]),
                                                                    _n1_short(got)), hint)
        for i in test.get("keep", []):
            if args[i] != kept[i]:
                return (head, "%s changed its input %s" % (name, _n1_short(kept[i])), _N1_KEEP_HINT[name])
        if test.get("table"):
            t = test["table"]
            if not _n1_same("table", args[t["arg"]], t["want"], test["tol"]):
                return (head, "after %s the table of bets should be %s, and it is %s" % (
                    name, _n1_short(t["want"]), _n1_short(args[t["arg"]])), _N1_KEEP_HINT[name])
    return None


def _n1_works(name):
    """True when the function called name exists, is finished, the functions it uses work, and it passes its tests.
    Remembered for each version of it."""
    fn = globals().get(name)
    if not callable(fn) or getattr(fn, "__code__", None) is None or _n1_has_gap(fn):
        return False
    for other in _N1_NEEDS.get(name, []):
        if _n1_uses(fn, other) and not _n1_works(other):
            return False
    if name not in _N1_TESTS:
        return True
    key = _n1_key(name)
    if key not in _N1_WORKING:
        _N1_WORKING[key] = _n1_failure(name, fn) is None
    return _N1_WORKING[key]


def _n1_waits(name, fn, cell):
    """One calm line and True when fn cannot run yet: it is not finished, or a function it uses does not work yet."""
    task = ""
    if name in _N1_TASKS:
        task = " (%s)" % _N1_TASKS[name]
    if _n1_has_gap(fn):
        print("⏳ %s is not finished yet%s: replace each ... with your code, run its cell, then %s again."
              % (name, task, cell))
        return True
    for other in _N1_NEEDS.get(name, []):
        if not _n1_uses(fn, other) or _n1_works(other):
            continue
        if other in _N1_TASKS:
            needed = globals().get(other)
            what = "does not work yet"
            if callable(needed) and _n1_has_gap(needed):
                what = "is not finished yet"
            print("⏳ %s uses your %s (%s), which %s. Get check_%s(%s) to ✅ first, then run %s again."
                  % (name, other, _N1_TASKS[other], what, other, other, cell))
            return True
        # a given function: it waits for the tasks it uses
        for deeper in _N1_NEEDS.get(other, []):
            if deeper in _N1_TASKS and not _n1_works(deeper):
                print("⏳ %s uses %s, which uses your %s (%s). Get check_%s(%s) to ✅ first, then run %s again."
                      % (name, other, deeper, _N1_TASKS[deeper], deeper, deeper, cell))
                return True
        print("⏳ %s uses %s, which does not work yet: run its cell again, then %s." % (name, other, cell))
        return True
    return False


def _n1_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _n1_waits(name, fn, "this check"):
        return
    failure = _n1_failure(name, fn)
    if failure is None:
        print("✅ " + _N1_SUCCESS[name])
        return
    head, what, hint = failure
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint + ".")


def _n1_safe_check(name, fn):
    try:
        _n1_check(name, fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_cumulative_reward(fn):
    """Task a: G1 of page 2's run and of short runs"""
    _n1_safe_check("cumulative_reward", fn)

def check_chances(fn):
    """Task b: the chances of page 3's rows of bets and of other rows"""
    _n1_safe_check("chances", fn)

def check_pick_action(fn):
    """Task c: the action of several numbers u, at the ends of the slices too"""
    _n1_safe_check("pick_action", fn)

def check_play_run(fn):
    """Task d: G1 of page 3's ten runs of the start, of a run after update 1 and of page 2's run"""
    _n1_safe_check("play_run", fn)

def check_average_reward(fn):
    """Task e: the average G1 of page 3's start, of its first 10 runs and after update 1"""
    _n1_safe_check("average_reward", fn)

def check_slope(fn):
    """Task f: nabla of page 3's updates 1 and 2, and of a hop of 0.5"""
    _n1_safe_check("slope", fn)

def check_update_bet(fn):
    """Task g: the new bet and the table after page 3's updates 1 and 2, and with alpha 0.5"""
    _n1_safe_check("update_bet", fn)

def check_proto_reinforce(fn):
    """Task h: the points and the table of page 3's first three updates, and of one update with alpha 0.02"""
    _n1_safe_check("proto_reinforce", fn)


def play(fn, *inputs):
    """Runs one of your functions and gives its answer: play(average_reward, bets, dice) is
    average_reward(bets, dice). When the function is not finished yet, uses a function that does not work yet, gets
    None from a cell above or stops with an error, play prints one calm line and gives None, so the cells below
    still run."""
    import time
    try:
        if not callable(fn):
            print("⏳ play needs your function itself, as in play(average_reward, bets, dice).")
            return None
        name = getattr(fn, "__name__", "your function")
        if _n1_waits(name, fn, "this cell"):
            return None
        for x in inputs:
            if x is None:
                print("⏳ %s waits for a value that a cell above did not give yet (its ⏳ line says why)." % name)
                return None
        if name in _N1_LONG and not _n1_works(name):
            print("⏳ %s waits until it passes its check: get check_%s(%s) to ✅ first, then run this cell again."
                  % (name, name, name))
            return None
        start = time.time()
        try:
            answer = fn(*inputs)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check says more." % (name, type(error).__name__, error))
            return None
        seconds = time.time() - start
        if seconds > 5:
            print("⏱️ %s took %d seconds." % (name, round(seconds)))
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the folded cells at the top again, then this cell." % error)
        return None

def _n1_ready(name, cell):
    """True when your function called name exists and passes its check; else one calm line."""
    fn = globals().get(name)
    if not callable(fn):
        print("⏳ %s is not there yet: run the cell of %s first, then %s." % (name, _N1_TASKS[name], cell))
        return False
    if _n1_waits(name, fn, cell):
        return False
    if not _n1_works(name):
        print("⏳ This cell waits until %s passes its check: get check_%s(%s) to ✅ first, then run %s again."
              % (name, name, name, cell))
        return False
    return True


def _n1_dollars(x):
    """-$185, $95, -$349.65"""
    if isinstance(x, int):
        text = "$%d" % abs(x)
    else:
        text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def show_roulette(row_of_chances, numbers):
    """Spins your pick_action once for every number: prints the first six spins, then how often each action came."""
    try:
        if row_of_chances is None or numbers is None:
            print("⏳ show_roulette waits for a row of chances from a cell above (its ⏳ line says why).")
            return
        if not _n1_ready("pick_action", "this cell"):
            return
        counts = [0, 0, 0]
        for k in range(len(numbers)):
            action = pick_action(row_of_chances, numbers[k])
            counts[action] = counts[action] + 1
            if k < 6:
                print("u = %.4f   ->  %s" % (numbers[k], ACTIONS[action]))
        print("...")
        print("In %s spins:  RUN %d times, SERVICE %d times, REPLACE %d times" % (
            format(len(numbers), ","), counts[0], counts[1], counts[2]))
        print("The chances:   RUN %.2f, SERVICE %.2f, REPLACE %.2f" % (
            row_of_chances[0], row_of_chances[1], row_of_chances[2]))
    except Exception as error:
        print("❌ show_roulette could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_run(policy, run_dice):
    """Plays one run with your play_run and prints it week by week: the state, the action, the money, then G1."""
    try:
        if policy is None or run_dice is None:
            print("⏳ show_run waits for a table of chances from a cell above (its ⏳ line says why).")
            return
        if not _n1_ready("play_run", "this cell"):
            return
        log = []
        the_week = globals()["week"]

        def week_written_down(state, action, u):
            next_state, money = the_week(state, action, u)
            log.append([state, action, next_state, money])
            return next_state, money
        globals()["week"] = week_written_down
        try:
            G1 = play_run(policy, run_dice)
        finally:
            globals()["week"] = the_week
        for k in range(len(log)):
            state, action, next_state, money = log[k]
            print("week %2d   %-8s %-8s %6s    next week %s" % (k + 1, STATES[state], ACTIONS[action],
                                                            _n1_dollars(money), STATES[next_state]))
        print("G1 = %s" % _n1_dollars(G1))
    except Exception as error:
        print("❌ show_run could not run (%s). Run the folded cells at the top again, then this cell." % error)

print("✅ The checks of notebook 1 are ready.")

## Old Bessie, one week at a time

The van of the arcade, as on page 1. Each week you choose one of three actions for the state the van is in:

- **RUN**: the route earns \$95 when the van is HEALTHY, \$72 when WORN, \$40 when SHAKY, \$16 when FAILING, and the van
  wears. In 2%, 8%, 28% or 55% of the weeks it breaks down: \$280 more in costs, and it is FAILING next week.
- **SERVICE**: a week in the garage, -\$50; the van comes back up to two levels better.
- **REPLACE**: a new van, -\$130; it is HEALTHY next week.

A run is one quarter of 13 weeks from HEALTHY. The kit plays a week for you: `week(state, action, u)` gives the
state of next week and the money of this week, and the number `u`, between 0 and 1, decides the luck. States and
actions are numbers, the rows and the columns of a table: `HEALTHY` is 0 and so on up to `FAILING`, 3; `RUN` is 0,
`SERVICE` 1, `REPLACE` 2.

In [ ]:
# One week of Old Bessie: in a state, with an action, the number u decides the luck
next_state, money = week(HEALTHY, RUN, 0.5)
print("HEALTHY and RUN, u = 0.5:   money", money, "   next week", STATES[next_state])

next_state, money = week(SHAKY, RUN, 0.1)
print("SHAKY and RUN, u = 0.1:     money", money, "   next week", STATES[next_state])

**Reading the result.** A HEALTHY van that runs earns \$95, and with `u` = 0.5 it is WORN next week. A SHAKY van that
runs with `u` = 0.1 breaks down: a small `u` falls in the slice of the breakdown, 28% of the numbers, so the week
brings \$40 minus \$280, that is -\$240, and a FAILING van.

## The dice

Page 3 compared tables of bets on the same luck: every average of its start and of updates 1 and 2 played the
same 1,000 runs. The kit keeps these runs as dice: `dice[run][week]` holds two numbers between 0 and 1. The first
picks the action, the second is the `u` of `week`, the van's luck.

In [ ]:
# The dice of page 3's start: 1,000 runs of 13 weeks, two numbers a week
dice = dice_of_update(0)
print(len(dice), "runs of", len(dice[0]), "weeks")
print("week 1 of run 1:", dice[0][0])
print("week 2 of run 1:", dice[0][1])

**Reading the result.** In week 1 of run 1, 0.2882 picks the action and 0.0757 is the luck. With the same dice, two tables
of bets meet the same luck week for week: whatever differs in their averages comes from the bets alone.

## 🎯 Task A. The cumulative reward G1

A run brings money week after week. $G_1$ adds it up, and each week counts 0.9 times as much as the week before,
as on page 2:

$$G_1 = r_1 + 0.9\, r_2 + 0.9^2\, r_3 + \dots + 0.9^{12}\, r_{13}$$

Write `cumulative_reward(rewards)`: a loop over the weeks with a discount that starts at 1 and shrinks by 0.9
every week. `GAMMA` is 0.9, from the kit.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# In the loop, money is this week's money. Add discount * money to total. Then shrink the discount for the
# next week: multiply it by GAMMA.

In [ ]:
#@title 💡 Hint 2 for task A: the two lines (double-click to open) { display-mode: "form" }
#         total = total + discount * money
#         discount = discount * GAMMA

In [ ]:
# 🎯 YOUR TURN, task A: the cumulative reward G1 of a run
def cumulative_reward(rewards):
    # rewards: the money of each week of a run, week 1 first.
    # Gives G1: the money of every week, each week counting 0.9 times as much as the week before.
    total = 0
    discount = 1                        # week 1 counts fully
    for money in rewards:
        total = ...                     # 🎯 YOUR TURN: add this week's money, times the discount
        discount = ...                  # 🎯 YOUR TURN: the next week counts GAMMA (0.9) times as much
    return total

In [ ]:
check_cumulative_reward(cumulative_reward)

💭 **Think first.** Page 2's run, RUN until it fails, earned \$184 in its 13 weeks, all added up plainly. Will its
$G_1$ be larger or smaller?

In [ ]:
# Page 2's run, RUN until it fails: the money of its 13 weeks
rewards = [95, 72, 40, 40, 40, -130, 95, -240, -130, 95, 95, 72, 40]
G1 = play(cumulative_reward, rewards)
print("G1 =", G1)

**Reading the result.** Smaller: $G_1$ = \$154.40, page 2's number in cents. The late weeks count less: the \$40 of week
13 counts as \$11.30, and the breakdown of week 8, -\$240, counts as -\$114.79.

## 🎯 Task B. From bets to chances: softmax

proto-REINFORCE starts from a table of bets picked at random: one row per state, one column per action. Softmax
turns each row of bets into a row of chances: e to the power of each bet, then each divided by their total.

$$\text{chance of RUN} = \frac{e^{w_{\text{RUN}}}}{e^{w_{\text{RUN}}} + e^{w_{\text{SERVICE}}} + e^{w_{\text{REPLACE}}}}$$

A higher bet gets a higher chance, every chance is above 0, and the chances of a row add up to 1. `exp(x)` from
the kit is e to the power x.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# Three small loops. The first fills powers with exp(bet), one power for each bet. The second adds the powers
# up into total. The third divides each power by total.

In [ ]:
#@title 💡 Hint 2 for task B: the three lines (double-click to open) { display-mode: "form" }
#         powers.append(exp(bet))
#         total = total + power
#         row_of_chances.append(power / total)

In [ ]:
# 🎯 YOUR TURN, task B: softmax, from a row of bets to a row of chances
def chances(row_of_bets):
    # row_of_bets: the bets of one state, one per action.
    # Gives the chances of the actions: e to the power of each bet, each divided by their total.
    powers = []
    for bet in row_of_bets:
        powers.append(...)              # 🎯 YOUR TURN: e to the power of the bet
    total = 0
    for power in powers:
        total = ...                     # 🎯 YOUR TURN: add up the powers
    row_of_chances = []
    for power in powers:
        row_of_chances.append(...)      # 🎯 YOUR TURN: this action's chance, its power divided by the total
    return row_of_chances

In [ ]:
check_chances(chances)

## The table of chances

Softmax of every row gives the table of chances: page 3 called it the probability table, the policy.
`table_of_chances` is given; it calls your `chances` once per state.

💭 **Think first.** SHAKY's bets are -0.40, 0.10, 10.00. How often will a SHAKY van be replaced?

In [ ]:
# Given: softmax of every row, the table of chances (page 3's probability table, the policy)
def table_of_chances(bets):
    # bets: the table of bets, one row per state. Gives the table of chances, one row per state.
    table = []
    for row_of_bets in bets:
        table.append(chances(row_of_bets))      # your softmax, one row at a time
    return table


bets = start_bets()                             # page 3's table of bets, picked at random
show_bets(bets)
print()
policy = play(table_of_chances, bets)           # the table of chances
show_chances(policy)

**Reading the result.** The extreme bet of SHAKY, 10.00, makes REPLACE almost certain: its chance is 0.9999, which the
table rounds to 1.00. WORN's extreme bet, -6.90, makes REPLACE almost impossible: 0.0011. HEALTHY's bets are close
together, and so are its chances: RUN 0.47, SERVICE 0.17, REPLACE 0.35. The last column names each state's likeliest action.

## 🎯 Task C. The roulette

Each week the van's state has its row of chances, and a number `u` between 0 and 1 picks the action, as the
roulette of page 3 did. Lay the chances end to end on the line from 0 to 1: with HEALTHY's chances, RUN takes the
numbers from 0 to 0.47, SERVICE from 0.47 to 0.65, REPLACE from 0.65 to 1. The action is the slice that `u` falls in.

Write `pick_action(row_of_chances, u)`: add the chances up one by one, and give back the first action whose
running total passes `u`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# running adds the chances up: running = running + row_of_chances[action]. After RUN it holds RUN's chance,
# after SERVICE the chances of RUN and SERVICE together.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# u falls in this action's slice when it is below the running total:
#         if u < running:

In [ ]:
# 🎯 YOUR TURN, task C: the roulette
def pick_action(row_of_chances, u):
    # row_of_chances: the chances of the actions of one state. u: a number between 0 and 1.
    # Gives the action whose slice u falls in: 0 for RUN, 1 for SERVICE, 2 for REPLACE.
    running = 0                                 # the chances added up so far
    for action in range(len(row_of_chances)):
        running = ...                           # 🎯 YOUR TURN: add this action's chance
        if ...:                                 # 🎯 YOUR TURN: u falls before the end of this action's slice
            return action
    return len(row_of_chances) - 1              # u at the very end of the line: the last action

In [ ]:
check_pick_action(pick_action)

In week 1 every run starts HEALTHY, so the first numbers of the 1,000 runs are 1,000 spins of HEALTHY's roulette.
`show_roulette` spins your `pick_action` on each of them.

In [ ]:
# The roulette of HEALTHY: in week 1 every run starts HEALTHY, and the week's first number picks the action
healthy = play(chances, bets[HEALTHY])          # HEALTHY's row of chances
numbers = []
for run_dice in dice:
    numbers.append(run_dice[0][0])              # week 1 of this run: the number that picks the action
show_roulette(healthy, numbers)

**Reading the result.** RUN comes 454 times, SERVICE 172 times and REPLACE 374 times: close to the chances times
1,000, as the luck of 1,000 spins allows.

## 🎯 Task D. A run: one quarter of 13 weeks

A run starts with a HEALTHY van. Each week the roulette of the state's row picks the action with the week's first
number, `week` plays the week with the second number, the money goes into the list of rewards, and the van moves
on to its next state. After 13 weeks, `cumulative_reward` gives the run's $G_1$.

`policy[state]` is the row of chances of a state: `policy[HEALTHY]` is HEALTHY's row.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# The roulette is your pick_action from task C. It takes the row of the state the van is in, policy[state],
# and the week's first number, numbers[0].

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
# At the end of the week the van is in next_state, so next week's roulette must use that row:
#         state = next_state

In [ ]:
# 🎯 YOUR TURN, task D: one run, a quarter of 13 weeks
def play_run(policy, run_dice):
    # policy: the table of chances. run_dice: the dice of one run, two numbers a week.
    # Plays 13 weeks from HEALTHY and gives the run's G1.
    state = HEALTHY                             # every run starts with a healthy van
    rewards = []                                # the money of each week
    for numbers in run_dice:                    # one week: numbers[0] picks the action, numbers[1] is the luck
        action = ...                            # 🎯 YOUR TURN: the roulette of this state's row, with numbers[0]
        next_state, money = week(state, action, numbers[1])    # the week happens
        rewards.append(money)
        state = ...                             # 🎯 YOUR TURN: next week starts where this week ends
    return cumulative_reward(rewards)

In [ ]:
check_play_run(play_run)

`show_run` plays run 1 of the dice with your `play_run` and the table of chances of the bets at random, and
prints it week by week.

In [ ]:
show_run(policy, dice[0])

**Reading the result.** Bets picked at random make a poor manager: in weeks 2, 8, 9 and 10 the van is HEALTHY and still goes
to the garage or gets replaced, and in week 3 it breaks down. Run 1 ends at $G_1$ = -\$349.65, page 3's first run, -\$350
in whole dollars. Run 2 earns about \$322: one run is one roll of luck.

## 🎯 Task E. The average G1 of 1,000 runs

The average over the 1,000 runs of the dice is a steady number: the average $G_1$, the number that proto-REINFORCE
raises.

$$\bar{G}_1 = \frac{G_1 \text{ of run 1} + G_1 \text{ of run 2} + \dots + G_1 \text{ of run 1000}}{1000}$$

Write `average_reward(bets, dice)`. It makes the table of chances once, plays every run of the dice, and divides
the total by the number of runs. The money is in cents, as on page 3: the last line rounds the average to 2
decimals.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task E (double-click to open) { display-mode: "form" }
# In the loop, play_run(policy, run_dice) is the G1 of one run, from task D: add it to total.

In [ ]:
#@title 💡 Hint 2 for task E (double-click to open) { display-mode: "form" }
# len(dice) is the number of runs:
#     average = total / len(dice)

In [ ]:
# 🎯 YOUR TURN, task E: the average G1 of many runs
def average_reward(bets, dice):
    # bets: the table of bets. dice: the dice of many runs.
    # Gives the average G1 of the runs, in cents.
    policy = table_of_chances(bets)             # softmax of every row
    total = 0
    for run_dice in dice:                       # one run
        total = ...                             # 🎯 YOUR TURN: add this run's G1 (task D)
    average = ...                               # 🎯 YOUR TURN: the total divided by the number of runs
    return round(average, 2)                    # in cents, as on page 3

In [ ]:
check_average_reward(average_reward)

In [ ]:
start = play(average_reward, bets, dice)
print("The average G1 of the bets at random:", start)

**Reading the result.** -\$172.24: page 3's start, the first point of its reward plot. On average the bets at random lose
money every quarter. The rest of the notebook changes the bets to raise this number.

## The hop: one small change of one bet

Which bet should change? Page 3 picked one bet at random: update 1 picked HEALTHY and RUN, a bet of 0.40. We add a hop of
0.01 to that bet in a copy of the table and play the same 1,000 runs again. Same dice, same luck: whatever changes in
the average comes from that bet alone.

💭 **Think first.** With 0.41 in place of 0.40, a HEALTHY van runs a little more often. Will the average $G_1$ go up or
down?

In [ ]:
# Update 1 of page 3, by hand
hop = 0.01                                      # the hop: a small change of one bet
bets = start_bets()                             # page 3's table of bets, picked at random
state, action = picked_bet(1)                   # the bet that update 1 picks at random
print("The bet picked at random:", STATES[state], "and", ACTIONS[action])

hopped = copy_bets(bets)                        # a copy: the table itself stays as it is
hopped[state][action] = hopped[state][action] + hop        # one hop up

before = play(average_reward, bets, dice)       # Before: the table as it is
after = play(average_reward, hopped, dice)      # After: the same runs, with the hopped table
print("Before:  the bet", round(bets[state][action], 2), "   the average G1", before)
print("After:   the bet", round(hopped[state][action], 2), "   the average G1", after)

**Reading the result.** Up, from -\$172.24 to -\$170.50: \$1.74 more. In a picture, with the bet across and the average $G_1$ up,
Before and After are two points, and the slope of the triangle between them, up divided by across, is nabla:

$$\nabla = \frac{\text{After} - \text{Before}}{\text{hop}} = \frac{-170.50 - (-172.24)}{0.01} = \frac{1.74}{0.01} = 174$$

## 🎯 Task F. The slope, nabla

Write `slope(bets, state, action, dice, hop)`: the steps of the cell above in a function. Before; a copy of the
table with the hop added to one bet; After, on the same dice; then up divided by across. The change is rounded to
cents and nabla to a whole number, as on page 3.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task F (double-click to open) { display-mode: "form" }
# The hop goes on the copy: hopped[state][action] = hopped[state][action] + hop. After is the average G1 of
# hopped, with the same dice: average_reward(hopped, dice).

In [ ]:
#@title 💡 Hint 2 for task F (double-click to open) { display-mode: "form" }
# Up is the change, across is the hop:
#     nabla = change / hop

In [ ]:
# 🎯 YOUR TURN, task F: the slope, nabla
def slope(bets, state, action, dice, hop):
    # How fast the average G1 rises when the bet of state and action rises, measured with a hop.
    # Gives nabla, a whole number, as on page 3.
    before = average_reward(bets, dice)         # Before: the table as it is
    hopped = copy_bets(bets)                    # a copy: the table itself stays as it is
    hopped[state][action] = ...                 # 🎯 YOUR TURN: the bet, one hop higher
    after = ...                                 # 🎯 YOUR TURN: After, the average G1 of the hopped table, same dice
    change = round(after - before, 2)           # up: the change of the average G1, in cents
    nabla = ...                                 # 🎯 YOUR TURN: the slope, up divided by across
    return round(nabla)

In [ ]:
check_slope(slope)

In [ ]:
nabla = play(slope, bets, state, action, dice, hop)      # update 1: HEALTHY and RUN
print("nabla =", nabla)

**Reading the result.** nabla = 174: near this bet, the average $G_1$ rises by \$174 for each unit of bet.

## 🎯 Task G. The update

A positive nabla says that raising the bet raised the average $G_1$. So the update raises the bet:

$$w = w + \alpha \nabla$$

In weekend 2 a loss had to go down: $w - \alpha \nabla$. Here the average $G_1$ has to go up: $w + \alpha \nabla$.
$\alpha = 0.01$ keeps each move small. Update 1: $w = 0.40 + 0.01 \times 174 = 2.14$.

Write `update_bet(bets, state, action, nabla, alpha)`. It puts the new bet into the table, in cents, and gives it
back.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task G (double-click to open) { display-mode: "form" }
# w is the bet as it is, alpha the step and nabla the slope: new_bet = w + alpha * nabla.

In [ ]:
#@title 💡 Hint 2 for task G (double-click to open) { display-mode: "form" }
# With w = 0.40, alpha = 0.01 and nabla = 174 the new bet is 0.40 + 1.74 = 2.14.

In [ ]:
# 🎯 YOUR TURN, task G: the update
def update_bet(bets, state, action, nabla, alpha):
    # The update of one bet, w = w + alpha nabla, in cents. Changes the table of bets and gives the new bet.
    w = bets[state][action]                     # the bet as it is
    new_bet = ...                               # 🎯 YOUR TURN: w plus alpha times nabla
    bets[state][action] = round(new_bet, 2)     # into the table, in cents, as on page 3
    return bets[state][action]

In [ ]:
check_update_bet(update_bet)

💭 **Think first.** With 2.14 in place of 0.40, softmax gives HEALTHY new chances: RUN is now 0.84. The 1,000 runs again,
with the same luck. Will the average $G_1$ go up or down?

In [ ]:
# Update 1 of page 3: the update, then the 1,000 runs again
alpha = 0.01                                    # alpha: keeps each move small
bets = start_bets()                             # page 3's table of bets, picked at random
new_bet = play(update_bet, bets, state, action, nabla, alpha)
print("The new bet of", STATES[state], "and", ACTIONS[action] + ":", new_bet)
print()
show_chances(play(table_of_chances, bets))
print()
print("The average G1 after update 1:", play(average_reward, bets, dice))

**Reading the result.** Up: from -\$172.24 to \$73.57, the second point of page 3's reward plot. One bet moved, and on
average the van now makes money.

## 🎯 Task H. proto-REINFORCE

Now the whole loop of page 3. Each update: a bet picked at random, the runs of that update, the slope of the bet,
its update, and one more point for the reward plot, the average $G_1$ after the update. The kit gives the bet of
each update, `picked_bet(number)`, and its dice, `dice_of_update(number)`, exactly as on page 3: the start's runs
for updates 1 and 2, fresh runs for every later update.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task H (double-click to open) { display-mode: "form" }
# The slope is your slope from task F, for the bet of this update: slope(bets, state, action, dice, hop).

In [ ]:
#@title 💡 Hint 2 for task H (double-click to open) { display-mode: "form" }
# The update is your update_bet from task G. It changes the table itself, so the line is the call alone:
#         update_bet(bets, state, action, nabla, alpha)

In [ ]:
# 🎯 YOUR TURN, task H: proto-REINFORCE
def proto_reinforce(bets, updates, hop, alpha):
    # bets: the table of bets, changed by every update. updates: how many updates.
    # Gives the points of the reward plot: the average G1 before the first update, then after each update.
    points = [average_reward(bets, dice_of_update(0))]     # the first point: the bets as they are
    for number in range(1, updates + 1):                   # update 1, 2, 3, ...
        state, action = picked_bet(number)                 # the bet picked at random, as on page 3
        dice = dice_of_update(number)                      # the runs of this update
        nabla = ...                                        # 🎯 YOUR TURN: the slope of that bet (task F)
        ...                                                # 🎯 YOUR TURN: the update of that bet (task G)
        points.append(average_reward(bets, dice))          # one more point: the average G1 after the update
    return points

In [ ]:
check_proto_reinforce(proto_reinforce)

## 300 updates

💭 **Think first.** Which action do you expect to be the likeliest in each state after 300 updates?

The next cell plays the 300 updates of page 3, each with three averages of 1,000 runs: about half a minute on Colab.

In [ ]:
# 300 updates of proto-REINFORCE, from page 3's bets at random (about half a minute)
bets = start_bets()
points = play(proto_reinforce, bets, 300, hop, alpha)

In [ ]:
plot_rewards(points)

**Reading the result.** The first point is the start, -\$172.24; after update 1 the average is \$73.57, and after update 300 it
is \$239.50, as on page 3. The row of dashes is the best strategy's average $G_1$ on the runs of update 300, \$244.03. From
update 3 on, every update plays fresh runs, a new roll of luck, so the points wobble a little. In 176 of the 300
updates nabla was 0: the hop of 0.01 changed no run. SHAKY's bet of REPLACE, 10.00, is one of them: it makes REPLACE
almost certain already, and it never moves.

In [ ]:
show_bets(bets)
print()
show_chances(play(table_of_chances, bets))

## The best strategy

After 300 updates the likeliest action in each state is RUN, SERVICE, REPLACE, REPLACE: the best strategy. Of the 81 strategies that take one
fixed action in each state, it has the highest average $G_1$ over a quarter, \$242.08, computed exactly from the van's
chances; on the runs of update 300 it gets \$244.03, the line of the plot. The table still takes another action now and
then: a FAILING van runs in 9 weeks of 100 and goes to the garage in 14. So the last point, \$239.50, sits a little
below the line.

## What you built

- **The forward pass**: `chances` turns bets into chances, `pick_action` spins the roulette, `play_run` plays a
  run and `cumulative_reward` gives its $G_1$, `average_reward` averages 1,000 runs.
- **The backward pass**: `slope` adds a hop of 0.01 to one bet, plays the same runs again and gives nabla.
- **The update**: `update_bet` moves the bet, $w = w + \alpha \nabla$.
- **The loop**: `proto_reinforce` repeats it, one bet picked at random each time, and draws the reward plot.

Next, notebook 2 runs the same algorithm on another game of the arcade, Last-Minute Pricing:
[proto-REINFORCE, notebook 2: Last-Minute Pricing](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_2_last_minute_pricing_student.ipynb).